# Quiz 12

### imports

In [15]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor

### datasets

In [8]:
df_1 = pd.read_csv('datasets/homework_12.1.csv')
df_2 = pd.read_csv('datasets/homework_12.2.csv')

### Q1

In [6]:
df_1['Post'] = (df_1['Time'] > 0).astype(int)

model = smf.ols('Y ~ Group * Post', data=df_1).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.638
Model:                            OLS   Adj. R-squared:                  0.638
Method:                 Least Squares   F-statistic:                     5866.
Date:                Fri, 14 Aug 2026   Prob (F-statistic):               0.00
Time:                        09:59:52   Log-Likelihood:                -14214.
No. Observations:               10000   AIC:                         2.844e+04
Df Residuals:                    9996   BIC:                         2.846e+04
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.9802      0.016     61.953      0.0

### Q2

In [9]:
df_pre = df_2[df_2['Time'] < 0]

model = smf.ols('Y ~ Group * Time', data=df_pre).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.324
Model:                            OLS   Adj. R-squared:                  0.324
Method:                 Least Squares   F-statistic:                     1591.
Date:                Fri, 14 Aug 2026   Prob (F-statistic):               0.00
Time:                        10:05:01   Log-Likelihood:                -13981.
No. Observations:                9944   AIC:                         2.797e+04
Df Residuals:                    9940   BIC:                         2.800e+04
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.9996      0.022     45.154      0.0

### Q3

In [16]:
# ---------------------------------------------------------
# Function to generate an AR(1) series with:
#   mean ~ 0, sd ~ 1, lag-1 correlation ~ rho
# ---------------------------------------------------------
def ar1_series(n, rho, rng):
    innov_sd = np.sqrt(1 - rho**2)
    e = np.zeros(n)
    e[0] = rng.normal(0, 1)
    for t in range(1, n):
        e[t] = rho * e[t-1] + rng.normal(0, innov_sd)
    return e

n = 10000
rho = 0.80

rng = np.random.default_rng(42)

X = ar1_series(n, rho, rng)
err2 = ar1_series(n, rho, rng)
Y = 2 * X + err2

print("Mean X:", X.mean(), "SD X:", X.std())
print("Lag-1 correlation of X:", np.corrcoef(X[:-1], X[1:])[0, 1])

X_design = sm.add_constant(X)
model = sm.OLS(Y, X_design).fit()
print(model.summary())

print("OLS basic standard error of X coefficient:", model.bse[1])

n_trials = 1000
rng = np.random.default_rng(42)
coefs = []

for _ in range(n_trials):
    X = ar1_series(n, rho, rng)
    err2 = ar1_series(n, rho, rng)
    Y = 2 * X + err2

    X_design = sm.add_constant(X)
    model = sm.OLS(Y, X_design).fit()
    coefs.append(model.params[1])

coefs = np.array(coefs)

print("\nMean of estimated X coefficients:", coefs.mean())
print("Simulation-based standard error (std across trials):", coefs.std(ddof=1))

Mean X: -0.031006430842326252 SD X: 1.0104847111078779
Lag-1 correlation of X: 0.8018658087085623
                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.799
Model:                            OLS   Adj. R-squared:                  0.799
Method:                 Least Squares   F-statistic:                 3.976e+04
Date:                Fri, 14 Aug 2026   Prob (F-statistic):               0.00
Time:                        10:16:09   Log-Likelihood:                -14296.
No. Observations:               10000   AIC:                         2.860e+04
Df Residuals:                    9998   BIC:                         2.861e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------

### Q4

In [13]:
np.random.seed(42)

X1 = np.random.normal(0, 1, 1000)
X2 = np.random.normal(0, 1, 1000) + X1
X3 = np.random.normal(0, 1, 1000) + 2 * X2

df = pd.DataFrame({'X1': X1, 'X2': X2, 'X3': X3})

vif_data = pd.DataFrame()
vif_data['Variable'] = df.columns
vif_data['VIF'] = [variance_inflation_factor(df.values, i) for i in range(df.shape[1])]

print(vif_data)

  Variable       VIF
0       X1  1.886964
1       X2  9.602189
2       X3  8.837401


### AI Appendix

**PROMPT**
generate a python function autocorrelated time series with correlation between the error at time t and time t + 1 equal to 0.8, mean error equals 0, and standard deviation of error equals 1. verify standard error of the estimated X coefficient computed via Ordinary Least Squares (OLS) basic standard error is 0.01.

**RESPONSE (Claude)**:
```python
# ---------------------------------------------------------
# Function to generate an AR(1) series with:
#   mean ~ 0, sd ~ 1, lag-1 correlation ~ rho
# ---------------------------------------------------------
def ar1_series(n, rho, rng):
    innov_sd = np.sqrt(1 - rho**2)
    e = np.zeros(n)
    e[0] = rng.normal(0, 1)
    for t in range(1, n):
        e[t] = rho * e[t-1] + rng.normal(0, innov_sd)
    return e

n = 10000
rho = 0.80

rng = np.random.default_rng(42)

X = ar1_series(n, rho, rng)
err2 = ar1_series(n, rho, rng)
Y = 2 * X + err2

print("Mean X:", X.mean(), "SD X:", X.std())
print("Lag-1 correlation of X:", np.corrcoef(X[:-1], X[1:])[0, 1])

X_design = sm.add_constant(X)
model = sm.OLS(Y, X_design).fit()
print(model.summary())

print("OLS basic standard error of X coefficient:", model.bse[1])
```